In [25]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

#### База знаний и первичный анализ

In [26]:
products_path = "data/products.csv"  
products_df = pd.read_csv(products_path)

len(products_df), products_df.head(3)


(25,
      product_id           sku                  product_name     category  \
 0  PRD_00000001   SF-00001-77          IronPath Pull-Up Bar       Sports   
 1  PRD_00000002  ESH-00002-77        NexLink Smart Doorbell  Electronics   
 2  PRD_00000003   EH-00003-68  Auraloop Over-Ear Headphones  Electronics   
 
   subcategory     brand   price    cost  stock_quantity  active  \
 0     Fitness  IronPath  148.20   43.91            1855    True   
 1  Smart Home   NexLink  151.56   80.15            1990    True   
 2  Headphones  Auraloop  217.62  113.81            2425    True   
 
             created_at  
 0  2024-08-16 05:50:22  
 1  2023-09-08 10:48:20  
 2  2023-05-17 19:40:35  )

В качестве базы знаний используем таблицу products.csv с товарами интернет‑магазина  
Каждый документ - один товар(название, категория, подкатегория, бренд, цена)  
По такой базе удобно будет строить retrieval/mini‑RAG для ответов на вопросы вида  
"какой есть йога‑мат", "какие есть наушники", "что есть для кемпинга" и тп

In [27]:
def product_to_text(row):
    return (
        f"product_id: {row['product_id']}. "
        f"Name: {row['product_name']}. "
        f"Category: {row['category']} / {row['subcategory']}. "
        f"Brand: {row['brand']}. "
        f"Price: {row['price']}."
    )

products_df["doc_text"] = products_df.apply(product_to_text, axis=1)
products_df[["product_id", "doc_text"]].head(5)


,product_id,doc_text
0,PRD_00000001,product_id: PRD_00000001. Name: IronPath Pull-...
1,PRD_00000002,product_id: PRD_00000002. Name: NexLink Smart ...
2,PRD_00000003,product_id: PRD_00000003. Name: Auraloop Over-...
3,PRD_00000004,product_id: PRD_00000004. Name: Elm Works Thro...
4,PRD_00000005,product_id: PRD_00000005. Name: Voltix Over-Ea...


#### Чанкинг документов

In [30]:
def simple_chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start = end - overlap
    return chunks

# Применим к одному документу для демонстрации
example_text = products_df["doc_text"].iloc[0]
simple_chunk_text(example_text, chunk_size=5, overlap=2)


['product_id: PRD_00000001. Name: IronPath Pull-Up',
 'IronPath Pull-Up Bar. Category: Sports',
 'Category: Sports / Fitness. Brand:',
 'Fitness. Brand: IronPath. Price: 148.2.']

In [31]:
chunks = []
for idx, row in products_df.iterrows():
    doc_id = row["product_id"]
    text = row["doc_text"]
    doc_chunks = simple_chunk_text(text, chunk_size=100, overlap=20)
    for i, ch in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc_id}_chunk_{i}",
            "product_id": doc_id,
            "chunk_text": ch
        })

chunks_df = pd.DataFrame(chunks)
len(chunks_df), chunks_df.head(5)


(25,
                chunk_id    product_id  \
 0  PRD_00000001_chunk_0  PRD_00000001   
 1  PRD_00000002_chunk_0  PRD_00000002   
 2  PRD_00000003_chunk_0  PRD_00000003   
 3  PRD_00000004_chunk_0  PRD_00000004   
 4  PRD_00000005_chunk_0  PRD_00000005   
 
                                           chunk_text  
 0  product_id: PRD_00000001. Name: IronPath Pull-...  
 1  product_id: PRD_00000002. Name: NexLink Smart ...  
 2  product_id: PRD_00000003. Name: Auraloop Over-...  
 3  product_id: PRD_00000004. Name: Elm Works Thro...  
 4  product_id: PRD_00000005. Name: Voltix Over-Ea...  )

Чанкинг сделала с chunk_size=100 и overlap=20, потому что описания товаров короткие и всё равно получался один чанк  
Эти параметры оставила, чтобы не пришлось переписывать функцию под длинные тексты  
В целом работает нормально, смысла усложнять не вижу

#### Эмбеддинги через TF‑IDF + NearestNeighbors
Поскольку мне не удалось выполнить этот пункт с `FAISS` (не подходит версия питона 3.13 для библиотеки sentence_transformers, а 3.10 качать было впадлу) мною было решено взять sklearn и TfidfVectorizer   
  
Построение TF‑IDF

In [32]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1
)

chunk_texts = chunks_df["chunk_text"].tolist()
X_tfidf = vectorizer.fit_transform(chunk_texts)  # shape: (n_chunks, n_features)
X_tfidf.shape


(25, 445)

NearestNeighbors

In [34]:
# Приведём к плотному float32 для FAISS
X_dense = X_tfidf.astype(np.float32).toarray()
nn_model = NearestNeighbors(
    n_neighbors=10,
    metric="cosine"
)
nn_model.fit(X_tfidf)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [ ]:
# функция поиска
def retrieve(query, top_k=5):
    q_vec = vectorizer.transform([query])
    distances, indices = nn_model.kneighbors(q_vec, n_neighbors=top_k)
    idxs = indices[0]
    sims = 1 - distances[0] 

    results = chunks_df.iloc[idxs].copy()
    results["similarity"] = sims
    return results


In [ ]:
# небольшая проверка
test_queries = [
    "yoga mat for fitness",
    "over-ear headphones",
    "camping lantern for outdoor",
    "air fryer for kitchen",
    "women's dress fashion"
]

for q in test_queries:
    print(f"\nQUERY: {q}")
    display(retrieve(q, top_k=3)[["chunk_id", "product_id", "chunk_text", "similarity"]])



QUERY: yoga mat for fitness


,chunk_id,product_id,chunk_text,similarity
23,PRD_00000024_chunk_0,PRD_00000024,product_id: PRD_00000024. Name: TrailRoot Yoga...,0.388455
0,PRD_00000001_chunk_0,PRD_00000001,product_id: PRD_00000001. Name: IronPath Pull-...,0.077574
17,PRD_00000018_chunk_0,PRD_00000018,product_id: PRD_00000018. Name: IronPath Pull-...,0.074197



QUERY: over-ear headphones


,chunk_id,product_id,chunk_text,similarity
2,PRD_00000003_chunk_0,PRD_00000003,product_id: PRD_00000003. Name: Auraloop Over-...,0.442704
4,PRD_00000005_chunk_0,PRD_00000005,product_id: PRD_00000005. Name: Voltix Over-Ea...,0.428802
10,PRD_00000011_chunk_0,PRD_00000011,product_id: PRD_00000011. Name: SoundMint Over...,0.425227



QUERY: camping lantern for outdoor


,chunk_id,product_id,chunk_text,similarity
22,PRD_00000023_chunk_0,PRD_00000023,product_id: PRD_00000023. Name: NordTrek Campi...,0.410650
19,PRD_00000020_chunk_0,PRD_00000020,product_id: PRD_00000020. Name: WildPath Campi...,0.378549
21,PRD_00000022_chunk_0,PRD_00000022,product_id: PRD_00000022. Name: NordTrek Hikin...,0.082316



QUERY: air fryer for kitchen


,chunk_id,product_id,chunk_text,similarity
20,PRD_00000021_chunk_0,PRD_00000021,product_id: PRD_00000021. Name: CopperNest Air...,0.41484
0,PRD_00000001_chunk_0,PRD_00000001,product_id: PRD_00000001. Name: IronPath Pull-...,0.00000
2,PRD_00000003_chunk_0,PRD_00000003,product_id: PRD_00000003. Name: Auraloop Over-...,0.00000



QUERY: women's dress fashion


,chunk_id,product_id,chunk_text,similarity
8,PRD_00000009_chunk_0,PRD_00000009,product_id: PRD_00000009. Name: Fern & Finch M...,0.288808
9,PRD_00000010_chunk_0,PRD_00000010,product_id: PRD_00000010. Name: Fern & Finch S...,0.164061
24,PRD_00000025_chunk_0,PRD_00000025,product_id: PRD_00000025. Name: StrideCo Ankle...,0.089302


#### Контрольные запросы + оценка retrieval

In [37]:
control_queries = [
    {"query": "over-ear headphones", "expected_product": "PRD_00000003"},
    {"query": "smart doorbell for home", "expected_product": "PRD_00000002"},
    {"query": "yoga mat", "expected_product": "PRD_00000024"},
    {"query": "camping lantern", "expected_product": "PRD_00000020"},
    {"query": "air fryer", "expected_product": "PRD_00000021"},
    {"query": "midi dress", "expected_product": "PRD_00000009"},
    {"query": "ankle boots", "expected_product": "PRD_00000025"},
    {"query": "collagen powder supplement", "expected_product": "PRD_00000017"},
    {"query": "lipstick matte", "expected_product": "PRD_00000014"},
    {"query": "hiking backpack", "expected_product": "PRD_00000022"},
]

K = 5

eval_rows = []
for item in control_queries:
    q = item["query"]
    expected = item["expected_product"]
    res = retrieve(q, top_k=K)
    retrieved_products = res["product_id"].tolist()
    hit = int(expected in retrieved_products)
    rank = retrieved_products.index(expected) + 1 if expected in retrieved_products else None
    
    eval_rows.append({
        "query": q,
        "expected_source": expected,
        "retrieved_sources": ",".join(retrieved_products),
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

retrieval_eval_df = pd.DataFrame(eval_rows)
retrieval_eval_df


,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,over-ear headphones,PRD_00000003,"PRD_00000003,PRD_00000005,PRD_00000011,PRD_000...",1,1
1,smart doorbell for home,PRD_00000002,"PRD_00000002,PRD_00000019,PRD_00000021,PRD_000...",1,1
2,yoga mat,PRD_00000024,"PRD_00000024,PRD_00000001,PRD_00000003,PRD_000...",1,1
3,camping lantern,PRD_00000020,"PRD_00000023,PRD_00000020,PRD_00000001,PRD_000...",1,2
4,air fryer,PRD_00000021,"PRD_00000021,PRD_00000001,PRD_00000003,PRD_000...",1,1
5,midi dress,PRD_00000009,"PRD_00000009,PRD_00000001,PRD_00000003,PRD_000...",1,1
6,ankle boots,PRD_00000025,"PRD_00000025,PRD_00000001,PRD_00000003,PRD_000...",1,1
7,collagen powder supplement,PRD_00000017,"PRD_00000017,PRD_00000001,PRD_00000003,PRD_000...",1,1
8,lipstick matte,PRD_00000014,"PRD_00000016,PRD_00000014,PRD_00000001,PRD_000...",1,2
9,hiking backpack,PRD_00000022,"PRD_00000022,PRD_00000001,PRD_00000003,PRD_000...",1,1


Подсчёт `hit@k` и `recall@k`

In [38]:
hit_at_k = retrieval_eval_df["hit_at_k"].mean()
recall_at_k = hit_at_k  # один релевантный документ на запрос

hit_at_k, recall_at_k


(np.float64(1.0), np.float64(1.0))

In [ ]:
# сохраняем артефакт
artifacts_dir = "artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

retrieval_eval_path = os.path.join(artifacts_dir, "retrieval_eval.csv")
retrieval_eval_df.to_csv(retrieval_eval_path, index=False)
retrieval_eval_path


'artifacts\\retrieval_eval.csv'

#### Небольшой эксперимент с параметрами retrieval

Буду проверять top_k=3 и top_k=5

In [40]:
def evaluate_top_k(K):
    rows = []
    for item in control_queries:
        q = item["query"]
        expected = item["expected_product"]
        res = retrieve(q, top_k=K)
        retrieved_products = res["product_id"].tolist()
        hit = int(expected in retrieved_products)
        rows.append(hit)
    return np.mean(rows)

for K in [3, 5]:
    print(f"top_k={K}, hit@{K}={evaluate_top_k(K):.3f}")


top_k=3, hit@3=1.000
top_k=5, hit@5=1.000


В эксперименте с top_k=3 и top_k=5 качество оказалось одинаковым  
Наверное потому что база знаний довольно маленькая и запросы подобраны так, что нужный документ всегда попадает в топ  
Так что разницы между k=3 и k=5 в такой ситуации почти нет и retrieval работает одинаково 

#### Обновление базы знаний и переиндексация

In [ ]:
# до обновления - первые 15 товаров
base_df_before = products_df.iloc[:15].copy()

# после обновления - вся база
base_df_after = products_df.copy()


In [63]:
def build_chunks_from_products(df):
    rows = []
    for _, row in df.iterrows():
        doc_id = row["product_id"]
        text = row["doc_text"]
        doc_chunks = simple_chunk_text(text, chunk_size=100, overlap=20)
        for i, ch in enumerate(doc_chunks):
            rows.append({
                "chunk_id": f"{doc_id}_chunk_{i}",
                "product_id": doc_id,
                "chunk_text": ch
            })
    return pd.DataFrame(rows)

chunks_before = build_chunks_from_products(base_df_before)
chunks_after  = build_chunks_from_products(base_df_after)

chunks_before.shape, chunks_after.shape


((15, 3), (25, 3))

In [ ]:
X_before = vectorizer.transform(chunks_before["chunk_text"])
X_after  = vectorizer.transform(chunks_after["chunk_text"])


In [64]:
# NearestNeighbors
index_before = NearestNeighbors(n_neighbors=10, metric="cosine")
index_before.fit(X_before)

index_after = NearestNeighbors(n_neighbors=10, metric="cosine")
index_after.fit(X_after)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [65]:
def retrieve_with_index(query, chunks_df, index, top_k=5):
    q_vec = vectorizer.transform([query])
    distances, indices = index.kneighbors(q_vec, n_neighbors=top_k)
    idxs = indices[0]
    sims = 1 - distances[0]

    res = chunks_df.iloc[idxs].copy()
    res["similarity"] = sims
    return res


In [66]:
queries_update = [
    "ankle boots",
    "yoga mat",
    "green tea bags"
]

rows_update = []

for q in queries_update:
    before_res = retrieve_with_index(q, chunks_before, index_before, top_k=5)
    after_res  = retrieve_with_index(q, chunks_after,  index_after,  top_k=5)

    before_sources = ",".join(before_res["product_id"].tolist())
    after_sources  = ",".join(after_res["product_id"].tolist())

    changed = int(before_sources != after_sources)

    rows_update.append({
        "query": q,
        "before_retrieved_sources": before_sources,
        "after_retrieved_sources": after_sources,
        "changed": changed
    })

retrieval_before_after_df = pd.DataFrame(rows_update)
retrieval_before_after_df


,query,before_retrieved_sources,after_retrieved_sources,changed
0,ankle boots,"PRD_00000001,PRD_00000002,PRD_00000003,PRD_000...","PRD_00000025,PRD_00000001,PRD_00000003,PRD_000...",1
1,yoga mat,"PRD_00000001,PRD_00000002,PRD_00000003,PRD_000...","PRD_00000024,PRD_00000001,PRD_00000003,PRD_000...",1
2,green tea bags,"PRD_00000015,PRD_00000001,PRD_00000003,PRD_000...","PRD_00000015,PRD_00000001,PRD_00000003,PRD_000...",0


In [67]:
retrieval_before_after_df.to_csv(
    "artifacts/retrieval_before_after_update.csv",
    index=False
)

"artifacts/retrieval_before_after_update.csv"

'artifacts/retrieval_before_after_update.csv'

#### Mini-RAG и краткий анализ ошибок

In [68]:
chunks_full = chunks_after
index_full = index_after

In [69]:
def retrieve_full(query, top_k=5):
    q_vec = vectorizer.transform([query])
    distances, indices = index_full.kneighbors(q_vec, n_neighbors=top_k)
    idxs = indices[0]
    sims = 1 - distances[0]

    res = chunks_full.iloc[idxs].copy()
    res["similarity"] = sims
    return res


In [70]:
def answer_question(question, top_k=5):
    retrieved = retrieve_full(question, top_k=top_k)

    answer = "Relevant products:\n"
    for _, row in retrieved.iterrows():
        answer += f"- {row['product_id']}: {row['chunk_text']}\n"

    sources = ",".join(retrieved["product_id"].tolist())
    return answer, sources


In [71]:
rag_questions = [
    "Do you have any yoga mats?",
    "What over-ear headphones are available?",
    "Is there a camping lantern?",
    "Any women's dresses?",
    "Do you sell collagen supplements?"
]

rag_rows = []

for q in rag_questions:
    ans, src = answer_question(q, top_k=5)
    rag_rows.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": src
    })

rag_examples_df = pd.DataFrame(rag_rows)
rag_examples_df


,question,answer,retrieved_sources
0,Do you have any yoga mats?,Relevant products:\n- PRD_00000024: product_id...,"PRD_00000024,PRD_00000001,PRD_00000003,PRD_000..."
1,What over-ear headphones are available?,Relevant products:\n- PRD_00000003: product_id...,"PRD_00000003,PRD_00000005,PRD_00000011,PRD_000..."
2,Is there a camping lantern?,Relevant products:\n- PRD_00000023: product_id...,"PRD_00000023,PRD_00000020,PRD_00000001,PRD_000..."
3,Any women's dresses?,Relevant products:\n- PRD_00000009: product_id...,"PRD_00000009,PRD_00000010,PRD_00000001,PRD_000..."
4,Do you sell collagen supplements?,Relevant products:\n- PRD_00000017: product_id...,"PRD_00000017,PRD_00000001,PRD_00000003,PRD_000..."


In [72]:
rag_examples_df.to_csv("artifacts/rag_examples.csv", index=False)
"artifacts/rag_examples.csv"


'artifacts/rag_examples.csv'

Во всех этих примерах первый результат корректен, поэтому `hit@3` и `hit@5` получились 1.0  
Пограничные и неидеальные случаи  
1. Лишние нерелевантные товары в top‑k 
   Во всех примерах mini‑RAG в `retrieved_sources` помимо целевого товара (`PRD_00000024`, `PRD_00000003`, `PRD_00000017`) возвращает ещё 3–4 продукта, которые к запросу напрямую не относятся

2. Обновление базы знаний (before/after) 
   - Для запроса ankle boots до обновления в top‑5 вообще не было `PRD_00000025`, а после обновления он появился первым.    
   - Для yoga mat аналогично 
   - Для green tea bags результаты до и после совпали (обновление базы не всегда меняет выдачу)

3. Шум в контексте mini‑RAG
   Поскольку mini‑RAG просто берёт top‑k чанков, он наследует все ошибки retrieval:  
   - если в top‑k есть нерелевантные товары, они попадают в ответ;  
   - если база знаний неполная (как было до обновления), правильный товар может вообще не попасть в контекст

Формально метрики на контрольных запросах hit@3 и hit@5 получились 1.0, но:

- качество ответа зависит не только от того, «есть ли правильный товар в top‑k»,  
- важны также шум в контексте, полнота базы знаний и формулировка запроса.

Основные ограничения текущего решения:  
- TF‑IDF чувствителен к формулировке запроса и не понимает смысл,  
- mini‑RAG полностью зависит от качества retrieval и состава top‑k.
